# A1: Tokens, tensors, shapes, logits

**Goal:** follow one prompt through TinyLlama and understand every shape.

```
text --tokenizer--> ids (N,) --embedding--> (N, 2048) --22 decoder layers--> (N, 2048) --lm_head--> logits (N, 32000)
```

Key ideas:
1. The model never sees text, only integer token ids.
2. We keep ids as one flat 1-D tensor `(N,)`. This is the *flattened-token* convention the whole engine will use.
3. Row `i` of the logits scores the token that comes **after** position `i`. Generation only needs the last row.
4. Position `i` cannot see later tokens (causality). This is the reason a KV cache is valid (Part C).

In [ ]:
# Setup: clone or pull the repo, cd into it, put it on sys.path, install pytest.
import os, subprocess, sys

REPO_URL = "https://github.com/guptasameer112/tinyserve.git"
REPO_DIR = "/content/tinyserve"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"], check=True)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
import importlib
import tinyserve.basics
importlib.reload(tinyserve.basics)  # picks up repo edits after a git pull without restarting the kernel
from tinyserve.envinfo import collect_env

collect_env()

## 1. Load the model
`load_model` returns the tokenizer and the Hugging Face model in FP16 on the GPU. For now the HF model is our *reference*; from Part B we build our own and compare against it.

In [ ]:
import torch
from tinyserve.basics import load_model, encode, forward_logits, next_token_greedy, shape_report

DEVICE = "cuda"
tokenizer, model = load_model(device=DEVICE, dtype=torch.float16)
print(model.config)

## 2. Text to tokens
Notice the first id is BOS (`<s>`, id 1), added by the tokenizer. Words may split into pieces, and `▁` marks a leading space.

In [ ]:
PROMPT = "The capital of France is"
prompt_ids = encode(tokenizer, PROMPT, DEVICE)
print("ids:", prompt_ids.tolist())
print("tokens:", tokenizer.convert_ids_to_tokens(prompt_ids.tolist()))
print("shape:", tuple(prompt_ids.shape), "dtype:", prompt_ids.dtype, "device:", prompt_ids.device)

## 3. Shapes and sizes
- embedding output: `(N, 2048)`; logits: `(N, 32000)`.
- Weights: about 1.1B params x 2 bytes (FP16) = about 2.2 GB. Every decoded token must read all of them (Lesson A2).
- Logits for N tokens cost `N * 32000 * 2` bytes. For long prompts we will only compute the last row (Part D).

In [ ]:
report = shape_report(tokenizer, model, PROMPT)
for key, value in report.items():
    print(f"{key:>18}: {value}")
print(f"weights: {report['weight_bytes'] / 1e9:.2f} GB")

## 4. Logits: what the numbers mean
Logits are unnormalised scores over the 32000-token vocabulary. `softmax` turns them into probabilities. The **last row** predicts the next token.

In [ ]:
logits = forward_logits(model, prompt_ids)
print("logits:", tuple(logits.shape), logits.dtype)

def show_top_k(logits_row, k):
    probs = torch.softmax(logits_row.float(), dim=-1)
    top = probs.topk(k)
    for prob, token_id in zip(top.values.tolist(), top.indices.tolist()):
        print(f"{prob:7.3%}  id={token_id:<6} {tokenizer.convert_ids_to_tokens(token_id)!r}")

show_top_k(logits[-1], 5)
print("greedy next token:", tokenizer.convert_ids_to_tokens(next_token_greedy(logits)))

## 5. A first (slow) generation loop
Append the greedy token, run the model again on the **whole** sequence, repeat. This works but redoes all earlier work every step. Part C fixes it with a KV cache.

In [ ]:
from tinyserve.basics import generate_greedy_naive

generated = generate_greedy_naive(model, prompt_ids, max_new_tokens=20)
print(tokenizer.decode(generated, skip_special_tokens=True))

## 6. Causality
Append extra tokens and compare the logits of the original positions. They should not change. In FP16 on GPU the kernels for different lengths may round slightly differently, so we expect a tiny difference, not bit-equality. The FP32 CPU test in `tests/` checks this tightly.

In [ ]:
from tinyserve.basics import prefix_logit_diff

extra = encode(tokenizer, " Paris and the weather is nice", DEVICE)[1:]  # drop the BOS
print("max |diff| on shared prefix (FP16):", prefix_logit_diff(model, prompt_ids, extra))

## 7. Run the lesson tests
Includes the `slow` test that checks the real TinyLlama config (22 layers, hidden 2048, 32 query heads, 4 KV heads, head_dim 64).

In [ ]:
result = subprocess.run([sys.executable, "-m", "pytest", "tests/test_a1_basics.py", "-v"], capture_output=True, text=True)
print(result.stdout[-3000:], result.stderr[-1000:])

In [ ]:
import json
from tinyserve.envinfo import save_result

saved_path = save_result("A1_shapes", report)
print(saved_path)
print(saved_path.read_text())